## River Crossing
### Introduction
We looked at the [Wolf, goat and cabbage problem](https://en.wikipedia.org/wiki/Wolf,_goat_and_cabbage_problem) river crossing problem in the unit material. This problem is very simple to solve by hand, and the solution path is not very long, so for this activity we will use another famous river crossing puzzle.

The [missionaries and cannibals](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem) is another toy river crossing problem, and is well-known in the AI literature because it was famously used by [Saul Amarel (1968)](https://web.archive.org/web/20080308224227/http://www.cc.gatech.edu/~jimmyd/summaries/amarel1968-1.html) as an example of problem representation in AI. Versions of the game are known to be at least [1000 years old](https://en.wikipedia.org/wiki/Missionaries_and_cannibals_problem#History). The problem is also subject of Exercise 3.9 in Russell & Norvig (2016, 3rd ed.) where it is stated as follows (p. 115).

> "Three missionaries and three cannibals are on one side of a river, along with a boat that can hold one or two people. Find a way to get everyone to the other side without ever leaving a group of missionaries in one place outnumbered by the cannibals in that place."

#### Note On Problematic Theme
Before we move on, an important note. In the modern age, the theme of this problem is problematic. The concept inescapably evokes images of colonialism, and was conceived in a time when sadly it was not uncommon to make associations between Black people and “cannibals”. As we said in the unit material, these problems are often repeated with different themes but the same underlying rules. As it happens, this problem was previously more commonly known as the *“jealous husbands”* problem – the puzzle states that no woman can be left unsupervised without her husband also present. For hopefully obvious reasons, I do not find this much of an improvement. 

The extremely short history of AI is dominated by white male voices, and still is today. We will revisit this topic in week 8 when we talk about AI ethics, because as AI continues to spread at a rapid pace this is having a demonstrable impact on people's lives.

Today this puzzle variant is called “missionaries and cannibals” or “jealous husbands” in the AI textbooks; I hope in a number of years we'll have moved beyond these themes entirely. I think it is completely possible (and reasonable) to come up with toy puzzles that feature no element of inequality or violence at all.

For this activity we will stay in line with the textbook and use this common theme, but if you decide to swap out the “missionaries” and the “cannibals” for something else, or even just use abstract labels, then it will not affect the lesson.

#### Before You Start
If you have not already, then take a moment to try to solve this puzzle yourself before we move onto the search based solution.

### Search Based Solution
As you saw in the Tower of Hanoi example, it is possible to solve this kind of problem through uninformed search. The following diagram shows the complete search space of the missionaries and cannibals problem. The initial state is shown on the left and the goal state is all the way to the right. Missionaries are represented by black triangles and cannibals by red circles. Arrows represent state transitions and are labelled with actions, e.g., 2c represents the action of two cannibals crossing the river.

<br><br>
<center>
<figure>
<img src="resources/mc-search-space.png" width=600>
<figcaption>The complete search space of the missionaries and cannibals problem. Credit: <a href=http://www.aiai.ed.ac.uk/~gwickler/missionaries.html>Gerhard Wickler</a></figcaption>
</figure>
</center>
<br><br>

### Your Task
Your task is to write a Python program that solves the missionaries and cannibals problem using **breadth-first search**. The pseudo-code from Russel and Norvig (p. 82) is repeated again below.

<img src="resources/Breadth_first_search.png" width=60%>

Unlike the Towers of Hanoi puzzle, in this task you will have to write all of the supporting code from scratch, with some suggestions. You may wish to use the accompanying "Infrastructure for search algorithms" shown in Section 3.3.1 of the same book (see the unit reading list).

Specifically, you may define a `Node` class with attributes
 * `state`: the state in the state space to which the node corresponds;
 * `parent` (optional): the node in the search tree that generated this node;
 * `action` (optional): the action that was applied to the parent to generate the node;
 
and methods

 * `is_goal_state()`: check whether the Node is the goal state;
 * `get_child_node()`: given an action, return the resulting child state;
 * `is_valid_state()`: would the state result in missionaries getting eaten?
 
Once you have a functioning `Node` class you need to come up with data structures for your `frontier` and your set of  `explored` nodes. 

The next choices you have to make is how to represent states and actions. You may follow Saul Amarel's approach of  representing the current state by a simple vector $<a,b,c>$. The vector's elements $a,b,$ and $c$ represent the number of missionaries on the wrong side, the number of cannibals on the wrong side, and the number of boats on the wrong side, respectively. Since all missionaries, all cannibals, and the boat start on the wrong side, the vector is initialised to $<3,3,1>$. Actions are represented using vector subtraction/addition to manipulate the state vector. For instance, if one cannibal crossed the river, the vector $<0,1,1>$ would be subtracted from the state to yield $<3,2,0>$.

You could also use a single Python tuple to represent the state, at the cost of having to more manually implement the state manipulations.

So you can (but do not have to) use the following structure and define two classes `Node` and `Game`:

In [ ]:
# I recommend you to start coding in another cell below and to keep this cell as a reference.
# That way you can incrementally build --> debug --> build --> debug --> ... your classes  
# instead of trying to do it all at once.

# class Node:
#     def __init__(self, m_wrong_side, c_wrong_side, boat_wrong_side):
#         self.state = ...
    
#     def is_goal_state(self):
#         ...

#     def get_child_node(self, action):
#         ...

#     ...

        
# class Game:
#     def __init__(self):
#         self.initial_node = Node(m_wrong_side=3, c_wrong_side=3, boat_wrong_side=1)
#         ...
    
#     def breadth_first_search(self):
#         ...


In [ ]:
# I recommend you to start coding in another cell below and to keep this cell as a reference.
# That way you can incrementally build --> debug --> build --> debug --> ... your classes  
# instead of trying to do it all at once.

from collections import deque

class Node:
    # (missionaries_in_boat, cannibal_in_boat, boat which is always 1 since any crossing action needs the boat):
    ACTIONS = [(1,0,1), (2,0,1), (0, 1,1), (0,2,1), (1,1,1)]
    TOTAL = 3

    def __init__(self, m_wrong_side, c_wrong_side, boat_wrong_side, parent=None, action =None):
        # Use a tuple since tuples are hashable, membership checks later are quick
        self.state = (m_wrong_side, c_wrong_side, boat_wrong_side)
        self.parent = parent
        # the single move that was taken to produce this particular node:
        self.action = action 

    def is_valid_state(self):
        m, c, boat = self.state
        # ensure we stick to the possible number of people:
        if not (0 <= m <= self.TOTAL and 0 <= c <= self.TOTAL):
            return False
    
        # checking that cannibals don't outnumber missionaries on the wrong side:
        wrong_side_check = (m == 0) or (m >= c)

        # checking that cannibals don't outnumber missionaries on the right side:
        right_side_check = (self.TOTAL - m == 0) or (self.TOTAL - m >= self.TOTAL - c)

        return wrong_side_check and right_side_check

    def is_goal_state(self):
        return self.state == (0, 0 , 0)

    def get_child_node(self, action):
        # so if boat is 1, so on the wrong side, the sign is negative and we subtract
        # if boat is 0, so on the right side, then sign is positive so we add
        sign = 1 - 2*self.state[2]

        # for each of the elements from state add or subtract and action element to or from it:

        next_state = tuple(s + sign * a for s, a in zip(self.state, action))
        # * break up next_state into the required 3 arguments for node:
        return Node(*next_state, parent=self, action=action)

    def get_steps(self):
        node = self
        steps = []
        while node is not None:
            steps.append(node)
            node = node.parent

        steps.reverse()
        return steps

    def __str__(self):
        m, c, boat = self.state
        return f"Node <{m},{c},{boat}>"


class Game:
    def __init__(self):
        self.initial_node = Node(m_wrong_side=3, c_wrong_side=3, boat_wrong_side=1)
        self.generated = 0
        self.explored = 0

    
    def breadth_first_search(self, debug = False):
        node = self.initial_node
        if node.is_goal_state():
            return node
        # deque needs an iterable, so pass it a single-item list
        frontier = deque([node])
        # set copy of frontier as deque membership checks are slow while set checks are O(1)
        frontier_states = {node.state} 
        explored = set()

        while frontier:
            node = frontier.popleft()
            frontier_states.remove(node.state)
            # prints all the nodes including any dead
            if debug:
                print("Now Exploring", node, ".")
            explored.add(node.state)
            self.explored = len(explored)


            for action in Node.ACTIONS:
                child = node.get_child_node(action)
                self.generated += 1
                if not child.is_valid_state():
                    continue
                if child.state in explored or child.state in frontier_states:
                    continue
                if child.is_goal_state():
                    return child
                # add to the end of deque
                frontier.append(child)
                # add to the set
                frontier_states.add(child.state)


If you use the provided template you could then try to repduce the following printed output.

In [18]:
g = Game()
goal_node = g.breadth_first_search(debug=True)
print("The goal node is", goal_node)

print()

steps = goal_node.get_steps()
# Steps from start to end goal with no dead ends or duplicates:
print("Steps:", len(steps)-1)
# Every object get_child_node returned i.e.how many node objects were generated (including any duplicates or invalid children):
print("Generated:", g.generated)
# Explored = the states we popped and expanded:
print("Explored:", g.explored)

print()

# prints just the steps from start to the goal state (no dead ends included)
for n in steps:
    print(n, n.action)

Now Exploring Node <3,3,1> .
Now Exploring Node <3,2,0> .
Now Exploring Node <3,1,0> .
Now Exploring Node <2,2,0> .
Now Exploring Node <3,2,1> .
Now Exploring Node <3,0,0> .
Now Exploring Node <3,1,1> .
Now Exploring Node <1,1,0> .
Now Exploring Node <2,2,1> .
Now Exploring Node <0,2,0> .
Now Exploring Node <0,3,1> .
Now Exploring Node <0,1,0> .
Now Exploring Node <1,1,1> .
The goal node is Node <0,0,0>

Steps: 11
Generated: 65
Explored: 13

Node <3,3,1> None
Node <3,1,0> (0, 2, 1)
Node <3,2,1> (0, 1, 1)
Node <3,0,0> (0, 2, 1)
Node <3,1,1> (0, 1, 1)
Node <1,1,0> (2, 0, 1)
Node <2,2,1> (1, 1, 1)
Node <0,2,0> (2, 0, 1)
Node <0,3,1> (0, 1, 1)
Node <0,1,0> (0, 2, 1)
Node <1,1,1> (1, 0, 1)
Node <0,0,0> (1, 1, 1)


### Solution
The solution to this exercise will be made available through the course page once you submit your version. If you are completely stuck you can submit the file unfinished to see the solution – the submission is not graded, but you should try to get your version working.

### Advanced Extension
In the Tower of Hanoi example you implemented *depth first* search, which is similar to breadth first search. Thinking about the trade-offs between each algorithm, do you think depth first search would be effective for the Missionaries and Cannibals problem?

There is another technique called *iterative deepening* which combines the benefits of both algorithms. The basic idea is to repeatedly try a limited version of depth first *tree search* to ever increasing depths. First of all we try a depth first search limited to depth 0: this will simply be the root note. Then we try depth first search to a limit of 1, which is just the children of the root node. Then a limit of 2, and so on. We stop when we find the goal, or if we do not find any nodes with the given depth (in other words we have explored the entire graph). Because we are using the *tree* version of the algorithm (even on a graph problem), we do not need to store the `explored` set, and the memory requirements are far less.

You can read more about this technique in section 3.4.5 of Russell and Norvig – the section is on page 88 but I would recommend you start reading from page 86 which explains the limitations of depth-first search, the motivation for depth-limited search, and the natural conclusion: iterative deepening. 

This technique provides a excellent trade-off in terms of memory and time efficiency. It may seem wasteful to generate the early states multiple times, but there are not many states in the early part of the search tree. This means that iterative deepening is roughly the same as breadth first search for total computational time and has much better memory performance.

Here is the pseudocode from Russell and Norvig:

<br>
<center>
<img src="resources/iterative-deepening.png" width=60%>
</center>

**Task:** Try writing an iterative deepening solution to either the Missionaries and Cannibals problem or the Tower of Hanoi problem. Or, you could try another search technique from the textbook. Share your results on the forum!

In [26]:
CUTOFF = "cutoff" # the depth limit has been reached but there may be deeper nodes
FAILURE = None # this branch has been fully searched and no solution found

class IDS_GAME:
    def __init__(self):
        self.start_node = Node(m_wrong_side=3, c_wrong_side=3, boat_wrong_side=1)
        self.generated = 0

    def is_cycle(self, node):
        # check the node's own chain to avoid getting caught in a loop
        # walk up the chain till you reach the root e.g (3,3,1) could be a child we are on, then walk up to parent (3,2,0) so no match 
        # then walk up again and we hit the root at (3,3,1)  which does match so this prevents endless loops
        ancestor = node.parent
        while ancestor is not None:     # If parent is none we are at the root
            if ancestor.state == node.state:  
                return True
            # stepping up the chain, reassigning ancestor to ta higher parent
            ancestor = ancestor.parent
        return False # this state has not yet appeared on the chain of ancestors, leading to this child so no loop and we should explore this child

    def depth_limited_search(self, node, limit):
        #See Russell & Norvig 4th edition fig 3.12
        #We are using tree search, so we have no explored set
        if node.is_goal_state():
            return node
        if limit == 0:
            return CUTOFF       # we have reached depth limit but there could still be more levels below

        # we assume this subtree is fully explored within our budget and no goal found
        # If a child returns CUTOFF this disproves this and means we will explore again to one level deeper
        result = FAILURE     
        for action in Node.ACTIONS:
            child= node.get_child_node(action)
            self.generated +=1
            if not child.is_valid_state():
                continue
            if self.is_cycle(child):
                continue
            # the limit is how many more levels I can descend , so as we move down each level we need to subtract to use uop the budget
            child_result = self.depth_limited_search(child, limit -1)

            if child_result is CUTOFF:
                # something bellow this child hit the limit so this branch is not fully explored 
                # change the result from FAILURE to CUTOFF, but also keep exploring any remaining actions which may have the goal
                result = CUTOFF
            elif child_result is not FAILURE:
                return child_result # a goal node is found that is a solution
        return result   # CUTOFF if a child ran out of depth, otherwise FAILURE

    def iterative_deepening_search(self, max_depth=50, debug=False):
        #Russell & Norvig mention to loop from 0 to infinity but a max depth should help protect against possible hanging
        # There is repeated work between loops but this minimal cost allows us to only have to hold one path in memory
        for depth in range(max_depth):
            result = self.depth_limited_search(self.start_node, depth)
            if debug:
                print(f"Debug {depth}: {result}. generated so far: {self.generated}")
            # CUTOFF means we should explore deeper
            # FAILURE means there is no solution
            if result is not CUTOFF:
                return result
        return FAILURE


In [29]:
game_1 = IDS_GAME()
game_1_goal = game_1.iterative_deepening_search(debug = True)
print("The goal node is:", game_1_goal)
print()
steps = game_1_goal.get_steps()
print("Steps:", len(steps)-1)
print("Generated:", game_1.generated)

Debug 0: cutoff. generated so far: 0
Debug 1: cutoff. generated so far: 5
Debug 2: cutoff. generated so far: 25
Debug 3: cutoff. generated so far: 55
Debug 4: cutoff. generated so far: 105
Debug 5: cutoff. generated so far: 165
Debug 6: cutoff. generated so far: 235
Debug 7: cutoff. generated so far: 315
Debug 8: cutoff. generated so far: 405
Debug 9: cutoff. generated so far: 505
Debug 10: cutoff. generated so far: 615
Debug 11: Node <0,0,0>. generated so far: 661
The goal node is: Node <0,0,0>

Steps: 11
Generated: 661
